# Exploração inicial — Hemeroteca Digital Brasileira (HDB)

Primeiro contato com os dados para o projeto de **modelagem de tópicos em anúncios de pessoas escravizadas (séc. XIX)**.

**Observações sobre a fonte (verificado em nov/2026):**

- O domínio antigo `memoria.bn.br` **não existe mais** (NXDOMAIN). A HDB agora responde em **`memoria.bn.gov.br`**.
- A biblioteca [pyHDB](https://github.com/ericbrasiln/pyHDB) (Eric Brasil) tem `memoria.bn.br` fixo no código e depende de Selenium + Chrome, então **não funciona como está**. Aqui reaproveitamos a lógica dela (abrir o `docreader` com `bib` + termo e percorrer as ocorrências), mas usando apenas `requests`, sem navegador. A lógica vive no pacote `src/hdb/` (módulo `hdb.busca`); este notebook só a usa.
- A pyHDB, assim como este notebook, coleta **metadados** das ocorrências (acervo, ano, edição, página, link). **O texto OCR não é exposto**: a antiga exportação em PDF (`PDFExport.aspx`) agora retorna 404. Para a modelagem de tópicos ainda precisaremos de uma estratégia para obter o texto (OCR próprio das imagens, transcrição manual de uma amostra, outra base etc.).
- Seja gentil com o servidor: a BN está reabrindo o acervo aos poucos. Mantenha a pausa entre as requisições.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("..").resolve() / "src"))  # para importar o pacote hdb

from hdb.busca import coletar_ocorrencias, nova_sessao, periodo_legal, total_ocorrencias

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

sessao = nova_sessao()  # a pausa entre requisições está em hdb.config.PAUSA

## Funções de acesso ao DocReader

As funções (`abrir_busca`, `ler_ocorrencia`, `proxima_ocorrencia`, `total_ocorrencias`, `coletar_ocorrencias`) estão em [`src/hdb/busca.py`](../src/hdb/busca.py). Para o `docreader.aspx?bib=<acervo>&Pesq=<termo>`, avançar entre ocorrências é reenviar o formulário ASP.NET simulando o clique em "próxima ocorrência" (`OcorPosBtn`).

Em linha de comando: `PYTHONPATH=src python -m hdb buscar --bib 706744 --termo fugiu`.

> As funções que ficavam aqui foram movidas para `src/hdb/busca.py`.

## Teste rápido

Acervo pequeno (*Imperio do Brasil: Diario Fluminense*, 1825–1831) com o termo `fugiu`.

In [ ]:
coletar_ocorrencias(sessao, "706744", "fugiu", limite=5)

## Acervos e períodos

Usamos o **Jornal do Commercio (RJ)**, que tem um acervo por década de 1830 a 1889, o que permite comparar os anúncios entre os marcos legais.
O **Diário do Rio de Janeiro** fica como segunda série.

| Marco legal | Ano |
|---|---|
| Lei Feijó (proibição do tráfico, "lei para inglês ver") | 1831 |
| Lei Eusébio de Queirós (fim do tráfico atlântico) | 1850 |
| Lei do Ventre Livre | 1871 |
| Lei dos Sexagenários | 1885 |
| Lei Áurea | 1888 |

> Os códigos `bib` aparecem na URL do DocReader ao abrir um periódico no site da HDB. Acervos longos são divididos em `_01`, `_02` etc.

In [ ]:
ACERVOS = {
    "364568_02": ("Jornal do Commercio", "1830-1839"),
    "364568_03": ("Jornal do Commercio", "1840-1849"),
    "364568_04": ("Jornal do Commercio", "1850-1859"),
    "364568_05": ("Jornal do Commercio", "1860-1869"),
    "364568_06": ("Jornal do Commercio", "1870-1879"),
    "364568_07": ("Jornal do Commercio", "1880-1889"),
}

# Termos sugeridos na reunião; sem acento (recomendação da HDB).
# Expressões exatas vão entre aspas duplas, ex.: '"escravo fugido"'.
TERMOS = ["fugido", "fugiu", "escravo", "escrava", "preto", "preta", "aluga-se", "vende-se", '"ama de leite"']

## Contagem de ocorrências: termo × década

Só uma requisição por par (acervo, termo), então é rápido (~1–2 min).
**Atenção:** são contagens brutas de *páginas* com o termo. Décadas com mais edições digitalizadas terão mais ocorrências; para comparar, o ideal é normalizar pelo número de páginas do acervo (próximo passo).

In [ ]:
linhas = []
for bib, (jornal, decada) in ACERVOS.items():
    for termo in TERMOS:
        n = total_ocorrencias(sessao, bib, termo)
        linhas.append({"bib": bib, "jornal": jornal, "decada": decada, "termo": termo, "ocorrencias": n})
        print(f"{bib} {decada} {termo!r:>14}: {n}")

contagens = pd.DataFrame(linhas)
contagens["periodo_legal"] = contagens["decada"].str[:4].astype(int).map(periodo_legal)
contagens.to_csv(DATA_DIR / "contagens_termo_decada.csv", index=False)

In [ ]:
tabela = contagens.pivot_table(index="termo", columns="decada", values="ocorrencias", aggfunc="sum")
tabela

In [ ]:
ax = tabela.T.plot(marker="o", figsize=(10, 5), title="Jornal do Commercio — páginas com ocorrência do termo, por década")
ax.set_xlabel("década")
ax.set_ylabel("ocorrências (bruto)")
for ano in (1850, 1871, 1885):
    ax.axvline((ano - 1830) / 10, color="grey", ls="--", lw=0.8)

## Amostra de ocorrências (metadados + link)

Coleta as primeiras ocorrências de um termo num acervo. O link abre a página digitalizada na HDB, útil para leitura e para montar uma amostra de anúncios a transcrever.

In [ ]:
amostra = coletar_ocorrencias(sessao, "364568_04", "fugido", limite=15)
amostra.to_csv(DATA_DIR / "amostra_fugido_364568_04.csv", index=False)
amostra

## Próximos passos

1. **Texto dos anúncios:** a HDB não expõe o OCR. Avaliar: (a) baixar as imagens das páginas e rodar OCR (Tesseract/`pytesseract`, ou modelos mais recentes para impressos antigos), (b) transcrever manualmente uma amostra, (c) checar se a dissertação indicada pela Fernanda usa algum corpus já transcrito.
2. **Normalização:** obter o número total de páginas/edições por acervo para comparar as décadas.
3. **Mais acervos:** incluir o Diário do Rio de Janeiro (`094170_01`, `094170_02`) e jornais de outras províncias.
4. **Combinações de termos:** testar expressões exatas (`"escravo fugido"`, `"preta para alugar"`) e variações ortográficas da época (`escravo`/`escrava`, `fugio`).